# 04 — Day 4 Feature Extraction: TRILLsson Embeddings

**Why this is a separate notebook, not another section in `01_features.ipynb`:** Days 1–3's improvement work all lived as appended sections in the original two notebooks, run sequentially on one Kaggle account. Day 4 is being run in parallel on a second account while Day 3 (HuBERT) finishes on the first — a growing shared notebook doesn't support that (running just the new cells still requires re-running everything before them in a fresh kernel). This notebook is self-contained: it only needs the raw dataset, nothing from `01_features.ipynb`'s prior sections.

**Why TRILLsson:** Days 2–3 stayed within the same family of ideas — SSL speech models originally built for ASR-adjacent goals (predict masked phonetic content), varying only *which layer* or *which pretraining objective*. TRILLsson is a different category: distilled specifically to capture **paralinguistic** information (voice quality, health/emotion-adjacent signal) rather than linguistic content — and it's the exact representation family the strongest reference paper (Norbury et al., same 3-activity shift-worker protocol) used. See `PROJECT_PLAN.md`'s Improvement plan, Day 4.

**What this notebook does:** loads each of the 280 recordings, runs them through a frozen TRILLsson model, and caches the resulting 1024-dim embedding (TRILLsson pools over time internally — no manual mean/std pooling needed here, unlike wav2vec2/HuBERT) to `trillsson.parquet`. Evaluation happens in the companion notebook, `04_evaluate_trillsson.ipynb`.

## 0. Setup — Inputs This Notebook Needs

- **Dataset:** `Fatigue_data` (the same `project_data` dataset used everywhere else) — Add Input -> Datasets.
- **Model:** `google/trillsson` — Add Input -> Models -> search "trillsson" -> framework **TensorFlow2** -> pick a variation. **Recommended: the largest available variation** (labelled `5` if Kaggle's variations are numbered 1-5) — extraction here is a one-time forward pass over 280 short clips, not training, so size/speed isn't a real constraint; bigger means closer to the full (non-distilled) model's embedding quality. Fall back to a smaller variation only if the largest fails to load.
- **Internet:** On (Settings -> Internet) — needed for `pip install tensorflow_hub`.
- **GPU:** Optional but recommended (Settings -> Accelerator) — speeds up the TensorFlow forward pass, though it's not essential for a model this size on clips this short.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import librosa

# Same robust resolution strategy as 01_features.ipynb — search recursively
# rather than assume a fixed /kaggle/input/ depth.
DATA_DIR = next(Path("/kaggle/input").rglob("project_data"))
AUDIO_DIR = DATA_DIR / "audio"
LABELS_CSV = DATA_DIR / "labels.csv"

OUT_DIR = Path("/kaggle/working/features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16000

print(f"DATA_DIR   = {DATA_DIR}")
print(f"AUDIO_DIR  = {AUDIO_DIR}")
print(f"LABELS_CSV = {LABELS_CSV}")

In [ ]:
labels = pd.read_csv(LABELS_CSV)
missing = [f for f in labels["filename"] if not (AUDIO_DIR / f).exists()]
assert not missing, f"{len(missing)} filenames missing audio files: {missing[:5]}"
assert len(labels) == 280, f"expected 280 rows, got {len(labels)}"
print(f"Loaded {len(labels)} labeled recordings, all resolve to audio files.")

## 1. Locate the TRILLsson Model

Kaggle Models mount under `/kaggle/input/` just like datasets, but the exact subfolder depth/casing depends on which framework and variation you attached. Rather than hardcode a path (which broke once already for datasets — see `01_features.ipynb`'s setup), search for the one file that always marks a TensorFlow SavedModel directory: `saved_model.pb`.

In [ ]:
candidate_dirs = sorted({p.parent for p in Path("/kaggle/input").rglob("saved_model.pb")})
print("Found TensorFlow SavedModel director(y/ies):")
for d in candidate_dirs:
    print(" ", d)

assert candidate_dirs, (
    "No TRILLsson SavedModel found -- did you attach the Kaggle Model input "
    "(Add Input -> Models -> google/trillsson)?"
)
TRILLSSON_DIR = candidate_dirs[0]
if len(candidate_dirs) > 1:
    print(f"\nMultiple SavedModels found -- using the first: {TRILLSSON_DIR}. "
          f"If that's not the intended variation, pick a different index above.")
print(f"\nUsing: {TRILLSSON_DIR}")

## 2. Load TRILLsson & Probe Its Interface

Unlike wav2vec2/HuBERT (Hugging Face `transformers`, a well-documented Python API used identically across models), TRILLsson is a TensorFlow SavedModel loaded via `tensorflow_hub`. Its exact call signature (input shape, output dict key) follows a documented convention but isn't worth hardcoding blind. This section loads the model and runs it on **one** file first, printing exactly what comes back — so any mismatch is caught immediately on one file, not deep into a 280-file loop.

In [ ]:
!pip install -q tensorflow_hub

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

trillsson_model = hub.load(str(TRILLSSON_DIR))
print("Loaded TRILLsson.")
print("Callable signatures:", list(getattr(trillsson_model, "signatures", {}).keys()))

In [ ]:
# Probe on ONE file before committing to the full extraction loop.
sample_path = AUDIO_DIR / labels["filename"].iloc[0]
y, _ = librosa.load(sample_path, sr=SAMPLE_RATE, mono=True)
y_trimmed, _ = librosa.effects.trim(y, top_db=25)
audio_batch = y_trimmed.astype(np.float32)[np.newaxis, :]  # (1, T)

probe_output = trillsson_model(audio_batch)
print(type(probe_output))
if isinstance(probe_output, dict):
    for k, v in probe_output.items():
        print(f"  key={k!r}  shape={getattr(v, 'shape', None)}")
else:
    print("shape:", getattr(probe_output, "shape", None))

**Expected:** a dict containing an `'embedding'` key, shape `(1, 1024)`. **If the key or shape printed above differs, update `EMBEDDING_KEY` (and the squeeze logic if needed) in the next cell before running the full extraction loop** — this is the one part of Day 4 that wasn't verifiable without a live TF Hub session.

In [ ]:
EMBEDDING_KEY = "embedding"  # adjust if the probe above printed a different key


def extract_trillsson_embedding(wav_path: Path) -> np.ndarray:
    """Load one recording -> trim silence -> frozen TRILLsson forward pass ->
    the model's own internally-pooled 1024-dim embedding."""
    y, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)
    y_trimmed, _ = librosa.effects.trim(y, top_db=25)
    audio_batch = y_trimmed.astype(np.float32)[np.newaxis, :]  # (1, T)

    output = trillsson_model(audio_batch)
    embedding = np.asarray(output[EMBEDDING_KEY]).squeeze(0)  # (1024,)
    return embedding

## 3. Extract & Cache

In [ ]:
from tqdm.auto import tqdm

rows = []
for filename in tqdm(labels["filename"], desc="Extracting TRILLsson embeddings"):
    emb = extract_trillsson_embedding(AUDIO_DIR / filename)
    rows.append({"filename": filename, **{f"emb_{i}": v for i, v in enumerate(emb)}})

trillsson_df = pd.DataFrame(rows)
print(trillsson_df.shape)  # expect (280, 1025)
trillsson_df.head()

In [ ]:
out_path = OUT_DIR / "trillsson.parquet"
trillsson_df.to_parquet(out_path, index=False)

assert len(trillsson_df) == 280, f"expected 280 rows, got {len(trillsson_df)}"
n_emb_cols = sum(c.startswith("emb_") for c in trillsson_df.columns)
assert n_emb_cols == 1024, f"expected 1024 embedding dims, got {n_emb_cols}"
assert set(trillsson_df["filename"]) == set(labels["filename"]), "filename mismatch vs labels.csv"

print(f"Saved {out_path}")
print(f"Shape: {trillsson_df.shape} (280 rows x 1 filename + 1024 embedding dims)")
print("All verification checks passed.")

## Next steps

1. **Save this notebook's version** (Kaggle: "Save Version") so `trillsson.parquet` survives.
2. Open `04_evaluate_trillsson.ipynb`, attach **both** this notebook's output and the `Fatigue_data` dataset, and run it — it evaluates this embedding under the identical nested LOSO/GroupKFold(6) protocol used everywhere else in the project, and compares it against the results already logged in `PROJECT_PLAN.md`.